In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import time

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from sklearn.feature_selection import SelectKBest, chi2

In [ ]:
DIRETORIO_BASE = "."

# --- Caminhos (espelhando o padrao do SHAP) ---
CAMINHO_RAIZ_PROJETO = DIRETORIO_BASE
CAMINHO_ARQUIVO = os.path.join(CAMINHO_RAIZ_PROJETO, "dados/mh1m_balanceadas.npz")
BASE_DADOS = os.path.join(CAMINHO_RAIZ_PROJETO, "dados")
BASE_SRC_CHI2 = os.path.join(CAMINHO_RAIZ_PROJETO, "src_chi2")  # ranking por grupo: src_chi2/{grupo}/
os.makedirs(BASE_SRC_CHI2, exist_ok=True)

# Orcamentos gerados pelo SHAP (Script 4) -> definem o K de cada dataset
ORC_INDIVIDUAIS = os.path.join(BASE_DADOS, "orcamento_shap_individuais.csv")
ORC_PERM_OPCODES = os.path.join(BASE_DADOS, "orcamento_shap_permissions_opcodes.csv")
ORC_TODAS = os.path.join(BASE_DADOS, "orcamento_shap_todas.csv")

# Acumuladores de tempo (salvos em CSV ao final)
tempos_ranking = []   # tempo de calculo do ranking chi2 por grupo
tempos_dataset = []   # tempo de montagem de cada dataset



In [ ]:
# Carrega o dataset original (nao embaralhado: selecao mexe so em colunas)
dados = np.load(CAMINHO_ARQUIVO, allow_pickle=True)
X = dados["data"]
y = dados["classes"]
colunas = dados["column_names"]
print(f"Dataset original: X={X.shape}, y={y.shape}, colunas={colunas.shape}")



In [ ]:
# --- Funcoes auxiliares ---

def indices_do_grupo(nome_grupo):
    """Retorna os indices das colunas pertencentes ao grupo (namespace)."""
    if nome_grupo == "permissions_opcodes":
        idx = [i for i, n in enumerate(colunas) if n.startswith("permissions::") or n.startswith("opcodes::")]
    elif nome_grupo == "todas":
        idx = list(range(len(colunas)))
    else:
        idx = [i for i, n in enumerate(colunas) if n.startswith(f"{nome_grupo}::")]
    return idx


def gera_ranking_chi2(nome_grupo):
    """Calcula o ranking chi-quadrado do grupo sobre o dataset INTEIRO e salva em src_chi2/{grupo}/.
    Mede e registra o tempo de calculo (em segundos). Retorna o DataFrame do ranking."""
    idx = indices_do_grupo(nome_grupo)
    X_grupo = X[:, idx].astype(np.int8)        # chi2 exige valores >= 0 (dados binarios)
    colunas_grupo = colunas[idx]

    t0 = time.perf_counter()
    selector = SelectKBest(score_func=chi2, k="all")  # k="all": so calcula scores, nao corta
    selector.fit(X_grupo, y.astype(np.int8))
    tempo_ranking_s = time.perf_counter() - t0

    scores = selector.scores_
    pvalues = selector.pvalues_

    ordem = np.argsort(np.nan_to_num(scores, nan=-np.inf))[::-1]
    df_rank = pd.DataFrame({
        "feature": colunas_grupo[ordem],
        "f_score": scores[ordem],
        "pvalues": pvalues[ordem],
    })

    pasta = os.path.join(BASE_SRC_CHI2, nome_grupo)
    os.makedirs(pasta, exist_ok=True)
    df_rank.to_csv(os.path.join(pasta, "ranking_features_fscore_pvalues.csv"), index=False)

    tempos_ranking.append({
        "grupo": nome_grupo,
        "n_features": len(df_rank),
        "n_amostras": X_grupo.shape[0],
        "tempo_ranking_s": tempo_ranking_s,
    })
    print(f"[{nome_grupo}] ranking chi2 em {tempo_ranking_s:.4f}s | {len(df_rank)} features")
    return df_rank


def le_ranking_chi2(nome_grupo):
    caminho = os.path.join(BASE_SRC_CHI2, nome_grupo, "ranking_features_fscore_pvalues.csv")
    return pd.read_csv(caminho)


def monta_e_salva_dataset(nomes_features, caminho_saida, rotulo):
    """Aplica a mascara sobre o dataset original e salva o .npz reduzido.
    Mede e registra o tempo de montagem (em segundos)."""
    t0 = time.perf_counter()

    nomes_arr = np.array(list(dict.fromkeys(nomes_features)))
    existe = np.isin(nomes_arr, colunas)
    if not existe.all():
        faltando = nomes_arr[~existe]
        print(f"  ATENCAO [{rotulo}]: {len(faltando)} feature(s) nao encontrada(s). Ex.: {faltando[:5]}")

    mask = np.isin(colunas, nomes_arr)
    X_red = X[:, mask]
    colunas_red = colunas[mask]

    np.savez_compressed(caminho_saida, data=X_red, classes=y, column_names=colunas_red)
    tempo_dataset_s = time.perf_counter() - t0

    print(f"  [{rotulo}] data={X_red.shape}, column_names={colunas_red.shape}")
    print(f"  [{rotulo}] salvo em: {caminho_saida} | tempo={tempo_dataset_s:.4f}s")

    chk = np.load(caminho_saida, allow_pickle=True)
    print(f"  [{rotulo}] verificacao -> data={chk['data'].shape}, classes={chk['classes'].shape}, column_names={chk['column_names'].shape}")

    tempos_dataset.append({
        "dataset": rotulo,
        "caminho": caminho_saida,
        "n_features": int(X_red.shape[1]),
        "tempo_montagem_s": tempo_dataset_s,
    })



In [ ]:
# ============================================================
# FASE 1 - Gerar ranking chi2 dos 6 grupos (sobre o dataset inteiro)
# (os datasets 2 e 3 precisam dos rankings dos grupos combinados)
# ============================================================
grupos_todos = ["intents", "permissions", "opcodes", "apicalls", "permissions_opcodes", "todas"]

for nome_grupo in tqdm(grupos_todos):
    gera_ranking_chi2(nome_grupo)
print("\nRankings chi2 dos 6 grupos gerados.")



In [ ]:
# ============================================================
# DATASET 1 - Individuais (uniao). top-K por grupo, K de orcamento_shap_individuais.csv
# ============================================================
print("=== DATASET 1: individuais (uniao) ===")
orc_ind = pd.read_csv(ORC_INDIVIDUAIS)
# mapeia grupo -> K (n_features_selecionadas pelo SHAP)
k_por_grupo = dict(zip(orc_ind["grupo"], orc_ind["n_features_selecionadas"]))
print("K por grupo (SHAP):", k_por_grupo)

features_uniao = []
for nome_grupo in ["intents", "permissions", "opcodes", "apicalls"]:
    df_rank = le_ranking_chi2(nome_grupo)
    k = int(k_por_grupo[nome_grupo])
    top_k = df_rank.head(k)["feature"].tolist()
    features_uniao.extend(top_k)
    print(f"  [{nome_grupo}] top-{k} selecionadas")

monta_e_salva_dataset(
    features_uniao,
    os.path.join(BASE_DADOS, "mh1m_balanceadas_chi2.npz"),
    "DS1 chi2 individuais",
)
print()



In [ ]:
# ============================================================
# Salva os tempos medidos (em segundos)
# ============================================================
df_tempos_rank = pd.DataFrame(tempos_ranking)
df_tempos_rank.to_csv(os.path.join(BASE_SRC_CHI2, "tempos_ranking_chi2.csv"), index=False)
print("Tempos de ranking salvos em:", os.path.join(BASE_SRC_CHI2, "tempos_ranking_chi2.csv"))
print(df_tempos_rank)

df_tempos_ds = pd.DataFrame(tempos_dataset)
df_tempos_ds.to_csv(os.path.join(BASE_SRC_CHI2, "tempos_montagem_datasets_chi2.csv"), index=False)
print("\nTempos de montagem salvos em:", os.path.join(BASE_SRC_CHI2, "tempos_montagem_datasets_chi2.csv"))
print(df_tempos_ds)


